In [264]:
import pandas as pd
import json

In [265]:
# load fgo json data
try:
    # 2. Open the file using the 'with' statement
    with open('./fantasyGO_data.json', 'r') as f:
        # 3. Load the JSON data from the file
        fgo_23_24 = json.load(f)

    # The 'data' variable now holds your JSON content
    # In Jupyter, placing the variable at the end of the cell will display it
    print("✅ JSON file loaded successfully!")

except FileNotFoundError:
    print(f"❌ Error: The file ./fantasyGO_data.json was not found. Please check the path.")
except json.JSONDecodeError:
    print(f"❌ Error: The file ./fantasyGO_data.json is not a valid JSON file. Please check its contents.")

✅ JSON file loaded successfully!


In [266]:
fgo_picks = []

# Loop through each league in the top-level list
for gameweek_data in fgo_23_24:
    gameweek_info = {
        'gameweek': gameweek_data.get('contest').split(' ')[1],
        'contestant_no': gameweek_data.get('contestant_no'),
        'prize_pool': gameweek_data.get('prize_pool')
    }

    # Loop through each page within the league (e.g., "page_1", "page_2")
    for i in range(1, 1000): # Assuming a max of 999 pages
        page_key = f'page_{i}'
        if page_key in gameweek_data:
            # Loop through each manager's entry on the page
            for entry_data in gameweek_data[page_key]:
                entry_info = {
                    'manager': entry_data.get('manager'),
                    'entry_num': entry_data.get('entry'),
                    'total_points': entry_data.get('points'),
                    'prize': entry_data.get('prize')
                }

                # Loop through each player pick in the entry
                for pick_data in entry_data.get('pick', []):
                    # Combine all the info into a single record
                    full_record = {
                        **gameweek_info,
                        **entry_info,
                        **pick_data
                    }
                fgo_picks.append({**gameweek_info, **entry_data})
        else:
            break # Stop if the next page doesn't exist

pd.DataFrame(fgo_picks).to_csv('./fgo_managers.csv', index=False)

In [267]:
# 2. Flatten the nested fgo_23_24
all_picks = []

# Loop through each league in the top-level list
for gameweek_data in fgo_23_24:
    gameweek_info = {
        'gameweek': gameweek_data.get('contest').split(' ')[1],
        'contestant_no': gameweek_data.get('contestant_no'),
        'prize_pool': gameweek_data.get('prize_pool')
    }
    # Loop through each page within the league (e.g., "page_1", "page_2")
    for i in range(1, 100): # Assuming a max of 99 pages
        page_key = f'page_{i}'
        if page_key in gameweek_data:
            # Loop through each manager's entry on the page
            for entry_data in gameweek_data[page_key]:
                entry_info = {
                    'manager': entry_data.get('manager'),
                    'entry_num': entry_data.get('entry'),
                    'total_points': entry_data.get('points'),
                    'prize': entry_data.get('prize')
                }

                # Loop through each player pick in the entry
                for pick_data in entry_data.get('pick', []):
                    # Combine all the info into a single record
                    full_record = {
                        **gameweek_info,
                        **entry_info,
                        **pick_data
                    }
                    all_picks.append(full_record)
        else:
            break # Stop if the next page doesn't exist

# 3. Create the DataFrame
fgo_df = pd.DataFrame(all_picks)

# Optional: Clean up the 'points' and 'total_points' columns
fgo_df['points'] = pd.to_numeric(fgo_df['points'])
fgo_df['total_points'] = fgo_df['total_points'].str.replace(' Points', '', regex=False).astype(float)

# --- Display the Result ---
print("✅ DataFrame created successfully!")
print(f"Total rows: {len(fgo_df)}")
print("\n--- Sample of the final DataFrame ---")

fgo_df

✅ DataFrame created successfully!
Total rows: 110088

--- Sample of the final DataFrame ---


,gameweek,contestant_no,prize_pool,manager,entry_num,total_points,prize,player_name,points,position,is_captin,is_vice
0,1,175,"R 8,180.18",Radynster,1,81.5,"R 5,596.88",Sanchez,2.0,,False,False
1,1,175,"R 8,180.18",Radynster,1,81.5,"R 5,596.88",Veltman,1.0,,False,False
2,1,175,"R 8,180.18",Radynster,1,81.5,"R 5,596.88",Wan-Bissaka,12.0,,False,False
3,1,175,"R 8,180.18",Radynster,1,81.5,"R 5,596.88",Estupiñan,7.0,,False,False
4,1,175,"R 8,180.18",Radynster,1,81.5,"R 5,596.88",Mitoma,5.0,,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
110083,38,256,"R 9,500.00",MJ23,2,26.0,,B.Fernandes,0.0,,True,False
110084,38,256,"R 9,500.00",MJ23,2,26.0,,Bailey,0.0,,False,False
110085,38,256,"R 9,500.00",MJ23,2,26.0,,Haaland,3.0,,False,True
110086,38,256,"R 9,500.00",MJ23,2,26.0,,Darwin,1.0,,False,False


In [268]:
fpl_players_df = pd.read_csv('../with new features/data/vaastav/data/2023-24/players_raw.csv')
fgo_names = fgo_df['player_name'].unique().tolist()
fpl_web_names = fpl_players_df['web_name'].unique().tolist()

In [269]:
# confirm names in fgo match those in fpl_web_names
match_df =[]
for name in fgo_names:
    if name in fpl_web_names:
        match_df.append({'name': name, 'has_match': True})
    else:
        match_df.append({'name': name, 'has_match': False})
match_df = pd.DataFrame(match_df)

match_df[~match_df['has_match']]['name'].to_list()

['Vinicius', 'Mitooma', 'Bradely', 'N.Semendo', 'De Bryune', 'Isak.']

In [270]:
fgo_missing_names_dict = {
    'Vinicius': 'Vinícius' ,
    'Mitooma': 'Mitoma',
    'Bradely': 'Bradley',
    'N.Semendo': 'N.Semedo',
    'De Bryune': 'De Bruyne',
    'Isak.': 'Isak'
}

# Use .map() to look up corrected names and .fillna() to keep original names that weren't in the dict.
fgo_df['player_name'] = fgo_df['player_name'].map(fgo_missing_names_dict).fillna(fgo_df['player_name'])

# Map: web_name -> element_type
position_map = fpl_players_df.set_index('web_name')['element_type'].to_dict()

# Map: web_name -> id
id_map = fpl_players_df.set_index('web_name')['id'].to_dict()

# This looks up each corrected player_name in your new dictionaries.
fgo_df['position'] = fgo_df['player_name'].map(position_map)
fgo_df['fpl_id'] = fgo_df['player_name'].map(id_map)

fgo_df.loc[fgo_df['fpl_id'] == 862, 'fpl_id'] = 204

In [271]:
# 1. Calculate the number of managers for each gameweek
# This calculates the size of each group and divides by 11
manager_counts = fgo_df.groupby('gameweek')['gameweek'].transform('size') / 11
fgo_df['managers'] = manager_counts

# 2. Calculate the ownership counts for each player within each gameweek
ownership_counts = fgo_df.groupby(['gameweek', 'player_name'])['player_name'].transform('size')
fgo_df['owned_by'] = ownership_counts

# 3. Calculate the ownership percentage in one go
# We need to get the manager count for each group to divide by
# manager_counts_for_calc = fgo_df.groupby(['gameweek', 'player_name'])['managers'].first()
ownership_percentage = round((fgo_df['owned_by']  / fgo_df['managers']) * 100, 2)
fgo_df['ownership_percent'] = ownership_percentage

fgo_df['score'] = fgo_df.apply(lambda row: row['points']/2 if row['is_captin'] else row['points']/1.5 if row['is_vice'] else row['points'], axis=1)
fgo_df

,gameweek,contestant_no,prize_pool,manager,entry_num,total_points,prize,player_name,points,position,is_captin,is_vice,fpl_id,managers,owned_by,ownership_percent,score
0,1,175,"R 8,180.18",Radynster,1,81.5,"R 5,596.88",Sanchez,2.0,1,False,False,145,66.0,1,1.52,2.0
1,1,175,"R 8,180.18",Radynster,1,81.5,"R 5,596.88",Veltman,1.0,2,False,False,151,66.0,9,13.64,1.0
2,1,175,"R 8,180.18",Radynster,1,81.5,"R 5,596.88",Wan-Bissaka,12.0,2,False,False,401,66.0,7,10.61,12.0
3,1,175,"R 8,180.18",Radynster,1,81.5,"R 5,596.88",Estupiñan,7.0,2,False,False,131,66.0,32,48.48,7.0
4,1,175,"R 8,180.18",Radynster,1,81.5,"R 5,596.88",Mitoma,5.0,3,False,False,143,66.0,16,24.24,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110083,38,256,"R 9,500.00",MJ23,2,26.0,,B.Fernandes,0.0,3,True,False,373,256.0,33,12.89,0.0
110084,38,256,"R 9,500.00",MJ23,2,26.0,,Bailey,0.0,3,False,False,34,256.0,6,2.34,0.0
110085,38,256,"R 9,500.00",MJ23,2,26.0,,Haaland,3.0,4,False,True,355,256.0,177,69.14,2.0
110086,38,256,"R 9,500.00",MJ23,2,26.0,,Darwin,1.0,4,False,False,293,256.0,4,1.56,1.0


## Combine with FPL merged data


In [272]:
fpl_23_24 = pd.read_csv('../with new features/data/joint/23-24/merged_player_data.csv')
players_preds_36 = pd.read_csv('../with new features/models/preds/players_preds_36.csv')
players_preds_37 = pd.read_csv('../with new features/models/preds/players_preds_37.csv')
players_preds_38 = pd.read_csv('../with new features/models/preds/players_preds_38.csv')
# fpl_23_24[['event', 'fpl_id', 'ownership_percent']]

In [273]:
# Select only the necessary columns
ownership_lookup = fgo_df[['gameweek', 'fpl_id', 'player_name','ownership_percent', 'score']].copy()

# Ensure 'gameweek' is numeric
ownership_lookup['gameweek'] = pd.to_numeric(ownership_lookup['gameweek'])

# Drop duplicate entries to have one row per player per gameweek
ownership_lookup.drop_duplicates(subset=['gameweek', 'fpl_id'], inplace=True)
ownership_lookup = ownership_lookup.rename(columns={'gameweek':'round'})

ownership_lookup[(ownership_lookup['round'] == 37) & (ownership_lookup['fpl_id'] == 362)]

,round,fpl_id,player_name,ownership_percent,score
104605,37,362,Palmer,84.77,14.0


In [274]:
missing_ids = set(set(ownership_lookup['fpl_id'].unique()) - set(fpl_23_24['fpl_id'].unique()))
print(len(missing_ids))

# Get the player names for missing ids
missing_players = ownership_lookup[ownership_lookup['fpl_id'].isin(missing_ids)].drop_duplicates(subset=['fpl_id'])
missing_players[['fpl_id', 'player_name']]

20


,fpl_id,player_name
6,505,Ndombele
33,498,Forster
627,323,Macey
704,299,Henderson
711,35,Buendia
714,443,Dennis
3408,207,Lukaku
5962,502,Lloris
30624,384,Heaton
30634,717,Stutter


In [275]:
# Merge on both 'gameweek' and 'fpl_id' to ensure correct matching
final_df = pd.merge(
    fpl_23_24,
    ownership_lookup,
    on=['round', 'fpl_id'],  # Use a list for multiple merge keys
    how='right'               # Use 'left' to keep all rows from merged_player_df
)


# Merge on both 'gameweek' and 'fpl_id' to ensure correct matching
preds_36 = pd.merge(
    players_preds_36,
    ownership_lookup,
    on=['round', 'fpl_id'],  # Use a list for multiple merge keys
    how='left'               # Use 'left' to keep all rows from merged_player_df
)

preds_37 = pd.merge(
    players_preds_37,
    ownership_lookup,
    on=['round', 'fpl_id'],  # Use a list for multiple merge keys
    how='left'               # Use 'left' to keep all rows from merged_player_df
)

preds_38 = pd.merge(
    players_preds_38,
    ownership_lookup,
    on=['round', 'fpl_id'],  # Use a list for multiple merge keys
    how='left'               # Use 'left' to keep all rows from merged_player_df
)

final_df['score_bps'] = final_df['score'] - final_df['bonus']
preds_36['score_bps'] = preds_36['score'] - preds_36['bonus']
preds_37['score_bps'] = preds_37['score'] - preds_37['bonus']
preds_38['score_bps'] = preds_38['score'] - preds_38['bonus']

final_df['ownership_percent'] = final_df['ownership_percent'].fillna(0)
preds_36['ownership_percent'] = preds_36['ownership_percent'].fillna(0)
preds_37['ownership_percent'] = preds_37['ownership_percent'].fillna(0)
preds_38['ownership_percent'] = preds_38['ownership_percent'].fillna(0)

final_df.dropna(subset=['fpl_name'], inplace=True)

In [276]:
final_df.to_csv('./fantasyGo_FPL.csv')
preds_36.to_csv('./fantasyGo_preds_36.csv')
preds_37.to_csv('./fantasyGo_preds_37.csv')
preds_38.to_csv('./fantasyGo_preds_38.csv')